# Module 2 — Efficient Data Processing with NumPy and Pandas
## From raw data to fast, reliable, memory-aware analysis

This module focuses on the tools data scientists use every day:
- **NumPy:** fast numerical arrays, vectorization, broadcasting, Boolean masks.
- **Pandas:** tabular data, cleaning, joins, groupby, reshaping, time series.
- **Efficiency:** avoid unnecessary Python loops, inspect memory, choose useful dtypes, and process larger files in chunks.
- **Reliability:** validate assumptions, handle missing data deliberately, avoid accidental mutation, and check output.

The examples use small synthetic data, so the notebook can run without downloading datasets.

## Learning outcomes

By the end, you should be able to:
1. Explain why NumPy arrays are different from Python lists.
2. Use vectorized calculations and broadcasting.
3. Filter and transform arrays with Boolean masks.
4. Create and inspect Pandas DataFrames.
5. Handle missing values, duplicates, types, and categories.
6. Use `groupby`, `merge`, and reshaping operations correctly.
7. Work with dates and build reusable aggregations.
8. Identify performance bottlenecks and reduce memory use.
9. Process a large CSV in chunks when the full file does not fit comfortably in memory.

# Part 1 — NumPy Foundations

## 1. Why NumPy?

A Python list can contain objects of different types. A NumPy array usually stores values of a consistent data type in a compact representation. This makes numerical operations efficient and enables vectorized calculations.

**Vectorization** means expressing an operation on an entire array rather than manually looping through each element in Python.

In [ ]:
import numpy as np

python_values = [1, 2, 3, 4]
numpy_values = np.array(python_values)

print("Array:", numpy_values)
print("Shape:", numpy_values.shape)
print("Dimensions:", numpy_values.ndim)
print("Data type:", numpy_values.dtype)

assert numpy_values.shape == (4,)
assert numpy_values.ndim == 1

## 2. Creating arrays and understanding shape

Common constructors:
- `np.array(...)`: convert values to an array.
- `np.zeros(...)` and `np.ones(...)`: initialize arrays.
- `np.arange(...)`: regularly spaced steps.
- `np.linspace(...)`: a fixed number of evenly spaced values.

Shape matters because many bugs are caused by arrays having dimensions different from what the programmer expects.

In [ ]:
zeros = np.zeros((2, 3))
ones = np.ones(4)
sequence = np.arange(0, 10, 2)
evenly_spaced = np.linspace(0, 1, 5)

assert zeros.shape == (2, 3)
assert ones.shape == (4,)
assert sequence.tolist() == [0, 2, 4, 6, 8]
assert np.allclose(evenly_spaced, [0, 0.25, 0.5, 0.75, 1])

## 3. Vectorized operations

Instead of looping through values to calculate a derived metric, apply the operation to the full array.

Suppose each value represents a paid claim amount. The array can calculate adjusted amounts or flags in one expression.

In [ ]:
paid = np.array([120.0, 85.5, 0.0, 240.0, 17.5])

adjusted_paid = paid * 1.05
is_high_cost = paid >= 100
positive_paid = paid[paid > 0]

assert np.allclose(adjusted_paid, [126.0, 89.775, 0.0, 252.0, 18.375])
assert is_high_cost.tolist() == [True, False, False, True, False]
assert positive_paid.tolist() == [120.0, 85.5, 240.0, 17.5]

### Exercise 1
Given `ages = np.array([18, 25, 41, 67, 15])`, create:
1. A Boolean mask for ages at least 18.
2. An array containing only ages at least 18.
3. An array of ages in years converted to months.

Try it before using the reference code.

In [ ]:
ages = np.array([18, 25, 41, 67, 15])
adult_mask = ages >= 18
adult_ages = ages[adult_mask]
age_months = ages * 12

assert adult_mask.tolist() == [True, True, True, True, False]
assert adult_ages.tolist() == [18, 25, 41, 67]
assert age_months.tolist() == [216, 300, 492, 804, 180]

## 4. Broadcasting

Broadcasting allows NumPy to combine compatible shapes without manually repeating data. Dimensions are compatible when they are equal or one of them is 1.

For example, subtracting a three-column mean vector from every row of a matrix centers each feature.

In [ ]:
measurements = np.array([
    [10.0, 100.0, 1.0],
    [20.0, 200.0, 2.0],
    [30.0, 300.0, 3.0],
])

column_means = measurements.mean(axis=0)
centered = measurements - column_means

assert column_means.shape == (3,)
assert centered.shape == (3, 3)
assert np.allclose(centered.mean(axis=0), [0, 0, 0])

**Remember:** `axis=0` usually means calculate down rows for each column; `axis=1` usually means calculate across columns for each row.

## 5. Missing values and safe numerical calculations

For floating-point NumPy arrays, missing numerical values are often represented by `np.nan`. Comparisons with `np.nan` behave unexpectedly: even `np.nan == np.nan` is false. Use `np.isnan`.

Be deliberate about whether a calculation should ignore, propagate, or reject missing values.

In [ ]:
values = np.array([10.0, np.nan, 30.0, 50.0])

mean_including_nan = np.mean(values)
mean_ignoring_nan = np.nanmean(values)
missing_mask = np.isnan(values)

assert np.isnan(mean_including_nan)
assert mean_ignoring_nan == 30.0
assert missing_mask.tolist() == [False, True, False, False]

## 6. Avoid unnecessary Python loops

Python loops are not always wrong. They are appropriate for complex control flow, but numerical operations over large arrays are often faster when expressed using NumPy operations.

Avoid using `np.vectorize` as a general performance fix: it is mainly a convenience wrapper and does not automatically turn Python code into compiled vectorized code.

In [ ]:
# Vectorized calculation of a simple ratio.
numerator = np.array([10.0, 20.0, 30.0])
denominator = np.array([2.0, 0.0, 5.0])

ratio = np.divide(
    numerator,
    denominator,
    out=np.full(numerator.shape, np.nan),
    where=denominator != 0
)

assert np.allclose(ratio[[0, 2]], [5.0, 6.0])
assert np.isnan(ratio[1])

# Part 2 — Pandas DataFrames

## 7. Creating and inspecting a DataFrame

A DataFrame is a labelled, two-dimensional table. Each column is a Series. The labels make it easier to join, filter, aggregate, and interpret data.

Useful first checks:
- `head()` and `sample()`
- `shape` and `columns`
- `dtypes`
- `info()`
- `isna().sum()`
- `duplicated().sum()`
- `describe()`

In [ ]:
import pandas as pd

claims = pd.DataFrame({
    "member_id": [101, 102, 101, 103, 104, 102, 105],
    "month": ["2026-01", "2026-01", "2026-02", "2026-01", "2026-02", "2026-02", "2026-02"],
    "product": ["A", "B", "A", "A", "C", "B", "C"],
    "medical_paid": [120.0, 250.0, 80.0, np.nan, 500.0, 100.0, 45.0],
    "pharmacy_paid": [20.0, 0.0, 35.0, 15.0, 70.0, 10.0, 5.0],
    "age": [34, 62, 34, 45, 71, 62, 28],
    "county": ["North", "South", "North", "East", "West", "South", "West"],
})

print(claims.head())
print("Shape:", claims.shape)
print("Types:\n", claims.dtypes)
print("Missing values:\n", claims.isna().sum())

assert claims.shape == (7, 7)

## 8. Selecting columns and filtering rows

Prefer explicit column selection and clear Boolean conditions. Use parentheses around each condition when combining masks with `&` or `|`; do not use Python's `and` or `or` on whole Series.

In [ ]:
selected = claims[["member_id", "medical_paid", "pharmacy_paid"]]
high_medical = claims[claims["medical_paid"] >= 100]
north_or_west = claims[claims["county"].isin(["North", "West"])]

assert selected.shape[1] == 3
assert set(high_medical["member_id"]) == {101, 102, 104}
assert set(north_or_west["county"]) == {"North", "West"}

### Exercise 2
Select rows where `age >= 40` and `pharmacy_paid > 0`. Return only `member_id`, `age`, and `pharmacy_paid`.

In [ ]:
exercise_result = claims.loc[
    (claims["age"] >= 40) & (claims["pharmacy_paid"] > 0),
    ["member_id", "age", "pharmacy_paid"]
]

assert list(exercise_result.columns) == ["member_id", "age", "pharmacy_paid"]
assert exercise_result["member_id"].tolist() == [103, 104, 102]

## 9. Creating columns safely

Use vectorized expressions for derived columns. For row-wise conditions, `np.where` or `np.select` is often clearer and faster than `DataFrame.apply(axis=1)`.

First decide how missing values should affect the derived metric.

In [ ]:
claims = claims.copy()
claims["total_paid"] = claims["medical_paid"].fillna(0) + claims["pharmacy_paid"]
claims["high_cost_flag"] = claims["total_paid"] >= 300

assert claims.loc[claims["member_id"] == 104, "total_paid"].iloc[0] == 570.0
assert claims["high_cost_flag"].dtype == bool

**Important:** Filling missing medical cost with zero is a business assumption in this example, not a universal rule. In real work, distinguish a true zero from a missing or unreported amount.

## 10. Missing values: inspect before filling

Common choices:
- Drop rows only when justified by the analysis.
- Fill with a domain-supported constant.
- Impute with a statistic or model when appropriate.
- Preserve a missingness indicator if missingness itself may carry information.

Never fill every missing column with zero without checking what missing means.

In [ ]:
missing_report = claims.isna().sum().sort_values(ascending=False)
claims_with_indicator = claims.assign(
    medical_paid_was_missing=claims["medical_paid"].isna()
)

assert missing_report["medical_paid"] == 1
assert claims_with_indicator["medical_paid_was_missing"].sum() == 1

## 11. Duplicates and data validation

A duplicate row is not always an error; it depends on the intended grain of the table. Decide what makes a record unique before dropping duplicates.

For example, if the intended grain is one record per member per month, validate that key.

In [ ]:
member_month_duplicates = claims.duplicated(subset=["member_id", "month"], keep=False)
duplicate_rows = claims.loc[member_month_duplicates, ["member_id", "month"]]

# This example intentionally has multiple claim records for some member-months.
assert len(duplicate_rows) == 0

# Validate a different table's unique key with an explicit check.
member_month = pd.DataFrame({
    "member_id": [101, 102, 103],
    "month": ["2026-01", "2026-01", "2026-01"],
})
assert not member_month.duplicated(["member_id", "month"]).any()

## 12. GroupBy: split, apply, combine

GroupBy divides data into groups, calculates a result for each group, and combines those results. Prefer built-in aggregations over custom row-wise functions when possible.

In [ ]:
monthly_product = (
    claims.groupby(["month", "product"], as_index=False)
    .agg(
        total_paid=("total_paid", "sum"),
        claim_rows=("member_id", "size"),
        unique_members=("member_id", "nunique"),
        average_paid=("total_paid", "mean"),
    )
)

assert {"total_paid", "claim_rows", "unique_members"}.issubset(monthly_product.columns)
assert monthly_product["claim_rows"].sum() == len(claims)

### Exercise 3
Calculate total paid and unique members by county, sorted from highest total paid to lowest.

In [ ]:
county_summary = (
    claims.groupby("county", as_index=False)
    .agg(
        total_paid=("total_paid", "sum"),
        unique_members=("member_id", "nunique")
    )
    .sort_values("total_paid", ascending=False)
    .reset_index(drop=True)
)

assert county_summary.iloc[0]["county"] == "West"
assert county_summary["unique_members"].max() >= 1

## 13. Merge / join DataFrames

A join combines tables using keys. Always understand the intended relationship:
- one-to-one;
- one-to-many;
- many-to-one;
- many-to-many.

A many-to-many join can multiply rows unexpectedly. Pandas supports `validate=` to check expected relationships and `indicator=True` to inspect unmatched keys.

In [ ]:
members = pd.DataFrame({
    "member_id": [101, 102, 103, 104, 105],
    "segment": ["Gold", "Silver", "Gold", "Bronze", "Silver"]
})

enriched = claims.merge(
    members,
    on="member_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert len(enriched) == len(claims)
assert (enriched["_merge"] == "both").all()

### Exercise 4
Change the join to `how="outer"` and inspect `_merge`. Add one member to `members` who has no claim row. Explain why the output row count can increase.

In [ ]:
members_extra = pd.concat([
    members,
    pd.DataFrame({"member_id": [999], "segment": ["New"]})
], ignore_index=True)

outer_join = claims.merge(
    members_extra,
    on="member_id",
    how="outer",
    validate="many_to_one",
    indicator=True
)

assert (outer_join["_merge"] == "right_only").any()

## 14. Reshaping: pivot and melt

- `pivot` turns unique key combinations into columns.
- `pivot_table` aggregates when multiple rows share the same keys.
- `melt` converts wide data into a long format.

Long format is often convenient for analysis and plotting.

In [ ]:
wide_costs = claims.pivot_table(
    index="member_id",
    columns="month",
    values="total_paid",
    aggfunc="sum",
    fill_value=0
)

long_costs = wide_costs.reset_index().melt(
    id_vars="member_id",
    var_name="month",
    value_name="total_paid"
)

assert wide_costs.index.name == "member_id"
assert {"member_id", "month", "total_paid"}.issubset(long_costs.columns)

## 15. Dates and time series

Convert date-like text into datetime values before doing date arithmetic. Invalid values can be set to `NaT`, but inspect how many conversions failed.

In [ ]:
date_data = pd.DataFrame({
    "date_text": ["2026-01-05", "2026-02-15", "invalid-date"]
})
date_data["date"] = pd.to_datetime(date_data["date_text"], errors="coerce")
date_data["year_month"] = date_data["date"].dt.to_period("M").astype("string")

assert date_data["date"].isna().sum() == 1
assert date_data.loc[0, "year_month"] == "2026-01"

## 16. Efficient transformations: vectorization before row-wise apply

A row-wise `apply` can be readable for complex logic, but it often has more Python overhead than a vectorized operation. Start with built-in Series operations, masks, `np.where`, and `groupby` aggregations.

In [ ]:
# Vectorized categorization
claims["cost_band"] = np.select(
    [
        claims["total_paid"] < 100,
        claims["total_paid"] < 300
    ],
    [
        "Low",
        "Medium"
    ],
    default="High"
)

assert set(claims["cost_band"]).issubset({"Low", "Medium", "High"})

# Part 3 — Performance and Memory

## 17. Measure before optimizing

Do not assume which line is slow. Measure runtime and memory, then optimize the actual bottleneck.

Useful tools:
- `time.perf_counter()` for elapsed time.
- `DataFrame.memory_usage(deep=True)` for approximate DataFrame memory.
- `df.info(memory_usage="deep")` for a compact overview.
- Profilers when timing alone does not reveal the hot path.

In [ ]:
import time

sample_values = np.arange(1_000_000)

start = time.perf_counter()
vectorized_result = sample_values * 2 + 5
elapsed_seconds = time.perf_counter() - start

assert vectorized_result[0] == 5
assert vectorized_result[-1] == (999_999 * 2 + 5)
print(f"Vectorized operation elapsed: {elapsed_seconds:.6f} seconds")

## 18. Inspect DataFrame memory

Object/string columns can consume substantial memory. Categorical dtype can help when a column repeats a relatively small set of labels. It is not automatically beneficial for every string column.

In [ ]:
memory_bytes = claims.memory_usage(deep=True).sort_values(ascending=False)
print(memory_bytes)

claims_optimized = claims.copy()
claims_optimized["product"] = claims_optimized["product"].astype("category")
claims_optimized["county"] = claims_optimized["county"].astype("category")

before_bytes = claims.memory_usage(deep=True).sum()
after_bytes = claims_optimized.memory_usage(deep=True).sum()

print("Before bytes:", before_bytes)
print("After bytes:", after_bytes)

assert claims_optimized["product"].dtype.name == "category"

## 19. Choose dtypes deliberately

Use the smallest safe dtype only after checking the value range and precision needs. Integer downcasting can save memory; floating-point downcasting can lose precision. Never trade away correctness just to reduce memory.

In [ ]:
numeric = pd.Series([0, 10, 200, 1000], dtype="int64")
downcast_numeric = pd.to_numeric(numeric, downcast="integer")

assert downcast_numeric.min() == numeric.min()
assert downcast_numeric.max() == numeric.max()

## 20. Chunked CSV processing

When a CSV is too large to fit in memory, process it in chunks. Aggregate each chunk and combine partial results. This is safe when the aggregation can be combined correctly; medians and some complex statistics need more careful strategies.

The following example creates an in-memory CSV string only to demonstrate the pattern. For a real file, pass its path to `pd.read_csv`.

In [ ]:
from io import StringIO

csv_text = StringIO(
    "county,paid\nNorth,10\nSouth,20\nNorth,5\nWest,7\nSouth,3\n"
)

partial_sums = {}
for chunk in pd.read_csv(csv_text, chunksize=2):
    chunk_sums = chunk.groupby("county")["paid"].sum()
    for county, value in chunk_sums.items():
        partial_sums[county] = partial_sums.get(county, 0) + value

assert partial_sums == {"North": 15, "South": 23, "West": 7}

## 21. Avoid common Pandas performance pitfalls

- Avoid repeated row-by-row DataFrame growth with `df.loc[len(df)] = ...`; collect records and build once, or concatenate a list of frames.
- Avoid `iterrows()` for numerical transformations when vectorized alternatives exist.
- Select only the columns you need when reading large files.
- Filter early when it preserves the required analysis.
- Use `groupby` built-in aggregations where possible.
- Be careful with joins that unexpectedly multiply rows.
- Prefer `.loc[...]` for explicit label-based assignment.
- Use `copy()` intentionally when independent data is required.

# Part 4 — Mini Project: Monthly Healthcare Cost Summary

Use the synthetic claim-level data to build a reproducible summary:
1. Create a total-paid metric with an explicit missing-value policy.
2. Group by month and product.
3. Calculate total cost, claim rows, and unique members.
4. Sort the summary and validate that aggregate row counts reconcile.
5. Explain what would change if missing medical cost were unknown rather than zero.

In [ ]:
def build_monthly_summary(claims_df: pd.DataFrame) -> pd.DataFrame:
    required = {"month", "product", "member_id", "medical_paid", "pharmacy_paid"}
    missing_columns = required - set(claims_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    result = claims_df.copy()
    # Explicit example policy: missing medical paid is treated as zero.
    result["total_paid"] = result["medical_paid"].fillna(0) + result["pharmacy_paid"]

    summary = (
        result.groupby(["month", "product"], as_index=False)
        .agg(
            total_paid=("total_paid", "sum"),
            claim_rows=("member_id", "size"),
            unique_members=("member_id", "nunique"),
        )
        .sort_values(["month", "product"])
        .reset_index(drop=True)
    )

    if summary["claim_rows"].sum() != len(result):
        raise AssertionError("Aggregation row counts do not reconcile")

    return summary


summary = build_monthly_summary(claims)
assert summary["claim_rows"].sum() == len(claims)
assert summary["total_paid"].sum() == claims["total_paid"].sum()
summary

# Final Practice Questions

1. Explain vectorization without using the phrase “it is faster” as your only explanation.
2. Explain the difference between `axis=0` and `axis=1`.
3. Why can filling missing costs with zero create misleading analysis?
4. When can a many-to-many join multiply rows?
5. Why is a vectorized expression often preferable to `apply(axis=1)`?
6. When is `category` dtype helpful, and when might it not help?
7. Why can chunked sum aggregation be combined across chunks but a median cannot generally be computed by taking the average of chunk medians?
8. What checks would you run before trusting a monthly cost summary?

# Readiness Checklist

- [ ] I can create arrays, inspect shapes, and use broadcasting.
- [ ] I can filter and transform data using vectorized operations.
- [ ] I can inspect data types, missing values, duplicates, and memory.
- [ ] I can use `groupby`, `merge`, `pivot_table`, and `melt`.
- [ ] I can explain the grain and cardinality of a join.
- [ ] I can justify a missing-value policy.
- [ ] I can process a large CSV in chunks when needed.
- [ ] I can test that aggregate results reconcile with the input.

**Next module:** SQL for data scientists — joins, CTEs, window functions, and analytical query patterns.